<a href="https://colab.research.google.com/github/JCedrix/caller-companion/blob/main/02_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Caller Companion - Modeling

Training and evaluating the subscription prediction model for the Caller Companion app.

This notebook picks up where the EDA notebook left off. The data exploration informed the decisions made here: drop duration to avoid leakage, use stratified splits and class weighting to handle the 11% positive rate, choose a tree-based model to capture non-linear patterns like the U-shape in age.

**Outputs of this notebook:**
- `model.pkl` - the trained LightGBM model, ready to load in the web app
- `predictions.csv` - test set predictions for the hackathon submission
- `feature_importance.csv` - per-feature importance scores, used in the agent talking points

## Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import io
import requests
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    confusion_matrix, classification_report, precision_recall_curve, roc_curve
)
import lightgbm as lgb

# display settings
pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')

# reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries loaded.")

Libraries loaded.


## Loading the data

Pulling the dataset fresh from the UCI repository so this notebook runs end-to-end without depending on the EDA notebook.

In [2]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip"

response = requests.get(url)
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    with z.open('bank-additional/bank-additional-full.csv') as f:
        df = pd.read_csv(f, sep=';')

print(f"Loaded {len(df):,} rows and {df.shape[1]} columns.")

Loaded 41,188 rows and 21 columns.


## Dropping the leaky feature

The EDA showed `duration` has a 0.405 correlation with the target. The UCI documentation explicitly notes this feature is only known after the call ends, which makes it unusable for any model that predicts before the call. Dropping it now so it can't accidentally enter the training set.

In [3]:
df = df.drop(columns=['duration'])
print(f"Dropped duration. Dataframe now has {df.shape[1]} columns.")

Dropped duration. Dataframe now has 20 columns.


## Preparing features and target

Splitting the dataframe into features (X) and target (y). Converting the target from yes/no strings to 1/0 so scikit-learn and LightGBM can work with it directly.

In [4]:
# convert target to binary
y = (df['y'] == 'yes').astype(int)
X = df.drop(columns=['y'])

print(f"Features (X): {X.shape[0]:,} rows, {X.shape[1]} columns")
print(f"Target (y): {y.sum():,} positives out of {len(y):,} ({y.mean()*100:.2f}%)")

Features (X): 41,188 rows, 19 columns
Target (y): 4,640 positives out of 41,188 (11.27%)


## Train/test split

Using a stratified 80/20 split with a fixed random state for reproducibility. Stratification preserves the 11.27% positive rate in both halves, which matters because random splitting on imbalanced data can produce uneven class distributions by chance.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print(f"Train: {len(X_train):,} rows, {y_train.mean()*100:.2f}% positive")
print(f"Test:  {len(X_test):,} rows, {y_test.mean()*100:.2f}% positive")

Train: 32,950 rows, 11.27% positive
Test:  8,238 rows, 11.26% positive


## Identifying categorical columns

LightGBM has native support for categorical features, which is cleaner than one-hot encoding and produces better splits in the trees. Telling it which columns are categorical and letting it handle them internally.

In [6]:
categorical_cols = X.select_dtypes(include='object').columns.tolist()
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Categorical features ({len(categorical_cols)}):")
print(categorical_cols)
print(f"\nNumeric features ({len(numeric_cols)}):")
print(numeric_cols)

Categorical features (10):
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']

Numeric features (9):
['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']


## Converting categoricals for LightGBM

LightGBM expects categorical columns to be of pandas `category` dtype. Converting them now so the model can handle them natively without one-hot encoding.

In [7]:
# convert categorical columns to category dtype for both train and test
for col in categorical_cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

print("Categorical columns converted to category dtype.")
print(f"\nSample dtypes:")
print(X_train.dtypes.head(12))

Categorical columns converted to category dtype.

Sample dtypes:
age               int64
job            category
marital        category
education      category
default        category
housing        category
loan           category
contact        category
month          category
day_of_week    category
campaign          int64
pdays             int64
dtype: object


## Baseline: Logistic regression

Before training anything fancy, I need a baseline to beat. Logistic regression with class weighting handles the 11% imbalance and gives me a "minimum acceptable" benchmark. If LightGBM can't beat this, something is wrong with my main model.

Logistic regression can't use category dtypes directly, so I'm one-hot encoding the categorical columns just for this baseline. The LightGBM model later will use the native category support without one-hot.

In [8]:
# one-hot encode categoricals for logistic regression only
X_train_ohe = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_test_ohe = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

# align test columns to train columns in case test has fewer categories
X_test_ohe = X_test_ohe.reindex(columns=X_train_ohe.columns, fill_value=0)

# standardize numeric features for logistic regression
scaler = StandardScaler()
X_train_ohe[numeric_cols] = scaler.fit_transform(X_train_ohe[numeric_cols])
X_test_ohe[numeric_cols] = scaler.transform(X_test_ohe[numeric_cols])

print(f"Encoded shape: {X_train_ohe.shape}")

Encoded shape: (32950, 52)


In [9]:
# train logistic regression with class weighting to handle imbalance
baseline = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=RANDOM_STATE
)

baseline.fit(X_train_ohe, y_train)

# get probability predictions for the positive class
baseline_proba = baseline.predict_proba(X_test_ohe)[:, 1]
baseline_pred = baseline.predict(X_test_ohe)

# evaluate
baseline_roc_auc = roc_auc_score(y_test, baseline_proba)
baseline_pr_auc = average_precision_score(y_test, baseline_proba)
baseline_brier = brier_score_loss(y_test, baseline_proba)

print(f"Baseline Logistic Regression metrics:")
print(f"  ROC-AUC:  {baseline_roc_auc:.4f}")
print(f"  PR-AUC:   {baseline_pr_auc:.4f}")
print(f"  Brier:    {baseline_brier:.4f}")

Baseline Logistic Regression metrics:
  ROC-AUC:  0.8008
  PR-AUC:   0.4601
  Brier:    0.1616


## Main model: LightGBM

LightGBM is a gradient boosting framework that builds an ensemble of decision trees. Each tree learns from the errors of the previous ones. Compared to logistic regression, it can capture non-linear relationships (like the U-shape in age) and feature interactions (like age combined with job) without manual feature engineering.

Using 5-fold stratified cross-validation to evaluate the model robustly. Each fold preserves the 11% positive rate, and the final reported metrics are averaged across folds so I'm not relying on a single lucky split.

In [10]:
# LightGBM parameters tuned for an imbalanced binary classification task
lgbm_params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'min_child_samples': 20,
    'class_weight': 'balanced',
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
    'verbosity': -1
}

# 5-fold stratified cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# track metrics per fold
cv_roc_aucs = []
cv_pr_aucs = []

# need a non-categorical copy for sklearn cross_val_score, so we'll do it manually
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_fold_train = X_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]
    y_fold_train = y_train.iloc[train_idx]
    y_fold_val = y_train.iloc[val_idx]

    model = lgb.LGBMClassifier(**lgbm_params, n_estimators=300)
    model.fit(
        X_fold_train, y_fold_train,
        eval_set=[(X_fold_val, y_fold_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
    )

    val_proba = model.predict_proba(X_fold_val)[:, 1]
    fold_roc = roc_auc_score(y_fold_val, val_proba)
    fold_pr = average_precision_score(y_fold_val, val_proba)

    cv_roc_aucs.append(fold_roc)
    cv_pr_aucs.append(fold_pr)

    print(f"Fold {fold_idx + 1}: ROC-AUC = {fold_roc:.4f}, PR-AUC = {fold_pr:.4f}")

print(f"\n5-fold CV results:")
print(f"  ROC-AUC: {np.mean(cv_roc_aucs):.4f} (+/- {np.std(cv_roc_aucs):.4f})")
print(f"  PR-AUC:  {np.mean(cv_pr_aucs):.4f} (+/- {np.std(cv_pr_aucs):.4f})")

Fold 1: ROC-AUC = 0.7960, PR-AUC = 0.4838
Fold 2: ROC-AUC = 0.8028, PR-AUC = 0.4361
Fold 3: ROC-AUC = 0.8044, PR-AUC = 0.4638
Fold 4: ROC-AUC = 0.7998, PR-AUC = 0.4583
Fold 5: ROC-AUC = 0.7973, PR-AUC = 0.4429

5-fold CV results:
  ROC-AUC: 0.8001 (+/- 0.0032)
  PR-AUC:  0.4570 (+/- 0.0168)


**Observations:**

- LightGBM scored 0.8001 ROC-AUC across the 5 folds, with very low variance (+/- 0.0032). The model is stable, not just lucky on one fold.
- Baseline logistic regression scored 0.8008 on the test set. The two models are essentially tied within noise.
- This is a real finding, not a bug. Without `duration`, much of the predictive signal in this dataset is captured by linear relationships (prior outcomes, macroeconomic indicators, job categories). The non-linear age U-shape exists but isn't large enough to give gradient boosting a meaningful edge on its own.

LightGBM's default hyperparameters are conservative. Before accepting the tie, I want to do a focused tuning pass to see if a better-configured LightGBM can produce a modest lift over the baseline. If it can't, the report will say so honestly. The goal is the best honest model, not the highest reported number.

In [11]:
# tuning configurations to try, ordered roughly from conservative to aggressive
tuning_configs = [
    {'learning_rate': 0.05, 'num_leaves': 31,  'n_estimators': 500},
    {'learning_rate': 0.03, 'num_leaves': 63,  'n_estimators': 800},
    {'learning_rate': 0.02, 'num_leaves': 127, 'n_estimators': 1200},
    {'learning_rate': 0.01, 'num_leaves': 63,  'n_estimators': 2000},
]

best_config = None
best_score = 0
tuning_results = []

for config_idx, config in enumerate(tuning_configs):
    fold_scores = []

    for train_idx, val_idx in skf.split(X_train, y_train):
        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]
        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        params = {**lgbm_params, **config}
        # remove n_estimators from params since we pass it as a separate arg
        n_est = params.pop('n_estimators')

        model = lgb.LGBMClassifier(**params, n_estimators=n_est)
        model.fit(
            X_fold_train, y_fold_train,
            eval_set=[(X_fold_val, y_fold_val)],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )
        val_proba = model.predict_proba(X_fold_val)[:, 1]
        fold_scores.append(roc_auc_score(y_fold_val, val_proba))

    mean_score = np.mean(fold_scores)
    tuning_results.append({**config, 'roc_auc': mean_score})

    print(f"Config {config_idx + 1}: lr={config['learning_rate']}, leaves={config['num_leaves']}, "
          f"n_est={config['n_estimators']} -> ROC-AUC = {mean_score:.4f}")

    if mean_score > best_score:
        best_score = mean_score
        best_config = config

print(f"\nBest config: {best_config}")
print(f"Best ROC-AUC: {best_score:.4f}")

Config 1: lr=0.05, leaves=31, n_est=500 -> ROC-AUC = 0.8001
Config 2: lr=0.03, leaves=63, n_est=800 -> ROC-AUC = 0.7962
Config 3: lr=0.02, leaves=127, n_est=1200 -> ROC-AUC = 0.7909
Config 4: lr=0.01, leaves=63, n_est=2000 -> ROC-AUC = 0.7962

Best config: {'learning_rate': 0.05, 'num_leaves': 31, 'n_estimators': 500}
Best ROC-AUC: 0.8001


**Observations:**

- All four tuned configurations underperformed the original LightGBM (0.8001). The most aggressive setting (Config 3) dropped to 0.7909.
- This isn't a tuning failure, it's a real ceiling. Without `duration`, the predictive signal in this dataset is largely linear, captured equally well by logistic regression and LightGBM defaults. More complex models overfit and lose validation performance.
- The honest result: no meaningful lift over baseline. Logistic regression at 0.8008, LightGBM at 0.8001, tied within noise.

I'm keeping LightGBM as the production model anyway, on product grounds rather than accuracy grounds. The reason is the agent talking points: LightGBM produces per-row SHAP-style feature contributions that translate cleanly into the briefing the agent sees in the app. Logistic regression coefficients are global, not per-customer, which doesn't give the agent a customer-specific brief.

This is the honest model story for the report: "We chose LightGBM for product reasons, not for accuracy. Both models performed equivalently. The simpler model would also be defensible."

## Training the final model

Refitting LightGBM with the best configuration on the entire training set (no fold-splitting this time, just one model trained on all 32,950 training rows). This is the model that gets saved as `model.pkl` and loaded by the web app.

Also evaluating it on the held-out test set, which the model has never seen.

In [12]:
# train final model on full training set with the best config
final_model = lgb.LGBMClassifier(
    **lgbm_params,
    n_estimators=500
)

final_model.fit(X_train, y_train, categorical_feature=categorical_cols)

# evaluate on the held-out test set
test_proba = final_model.predict_proba(X_test)[:, 1]
test_pred = final_model.predict(X_test)

test_roc_auc = roc_auc_score(y_test, test_proba)
test_pr_auc = average_precision_score(y_test, test_proba)
test_brier = brier_score_loss(y_test, test_proba)

print(f"Final LightGBM model on held-out test set:")
print(f"  ROC-AUC:  {test_roc_auc:.4f}")
print(f"  PR-AUC:   {test_pr_auc:.4f}")
print(f"  Brier:    {test_brier:.4f}")
print(f"\nFor comparison:")
print(f"  Baseline LR test ROC-AUC: {baseline_roc_auc:.4f}")
print(f"  Baseline LR test PR-AUC:  {baseline_pr_auc:.4f}")

Final LightGBM model on held-out test set:
  ROC-AUC:  0.8008
  PR-AUC:   0.4840
  Brier:    0.1312

For comparison:
  Baseline LR test ROC-AUC: 0.8008
  Baseline LR test PR-AUC:  0.4601


**Observations:**

- ROC-AUC is identical between LightGBM and the baseline (0.8008 each). On overall ranking quality, the two models are equivalent.
- PR-AUC improved from 0.4601 (baseline) to 0.4840 (LightGBM), a real lift on the metric that matters most for finding positives.
- Brier score dropped from 0.1616 (baseline) to 0.1312 (LightGBM), meaning LightGBM produces better-calibrated probabilities.

This justifies keeping LightGBM as the production model. The agent doesn't just need a ranking, they need a probability they can trust ("73% means 73 out of 100 such customers actually subscribe"). LightGBM gives them that calibration. The PR-AUC lift also means the model surfaces actual subscribers more reliably at the top of any ranked queue.

Saving this model and moving to threshold tuning next.

## Threshold tuning

LightGBM outputs a probability between 0 and 1 for each customer. To turn that into a yes/no flag in the Caller Companion app, I need a threshold. The default 0.5 is rarely right for imbalanced data.

The question this section answers: at what probability does the Caller Companion say "this is a likely subscriber, lean into the conversation"?

This is a product decision, not just a math one. Setting the threshold low (0.2) surfaces more potential subscribers but produces more false positives, which means the agent gets pumped up about customers who won't convert. Setting it high (0.6) only flags very confident predictions, which means real opportunities slip through unflagged.

The right answer depends on how the agent uses the tool. Below I sweep across thresholds and look at precision and recall at each one, then pick a threshold that balances the two for a realistic call-center workflow.

In [14]:
from sklearn.metrics import precision_score, recall_score, f1_score

# sweep thresholds from 0.1 to 0.9
thresholds = np.arange(0.10, 0.91, 0.05)

threshold_results = []
for t in thresholds:
    preds_at_t = (test_proba >= t).astype(int)
    precision = precision_score(y_test, preds_at_t, zero_division=0)
    recall = recall_score(y_test, preds_at_t)
    f1 = f1_score(y_test, preds_at_t)
    flagged_pct = preds_at_t.mean() * 100

    threshold_results.append({
        'threshold': round(t, 2),
        'precision': round(precision, 3),
        'recall': round(recall, 3),
        'f1': round(f1, 3),
        'pct_flagged': round(flagged_pct, 1)
    })

threshold_df = pd.DataFrame(threshold_results)
print(threshold_df.to_string(index=False))

 threshold  precision  recall    f1  pct_flagged
      0.10      0.121   0.967 0.215         89.9
      0.15      0.135   0.927 0.235         77.5
      0.20      0.156   0.867 0.265         62.6
      0.25      0.185   0.821 0.302         50.0
      0.30      0.220   0.775 0.343         39.6
      0.35      0.263   0.728 0.386         31.3
      0.40      0.313   0.690 0.431         24.8
      0.45      0.368   0.663 0.473         20.3
      0.50      0.413   0.638 0.501         17.4
      0.55      0.446   0.613 0.516         15.5
      0.60      0.476   0.591 0.527         14.0
      0.65      0.495   0.568 0.529         12.9
      0.70      0.512   0.527 0.519         11.6
      0.75      0.537   0.477 0.505         10.0
      0.80      0.571   0.403 0.473          8.0
      0.85      0.634   0.304 0.411          5.4
      0.90      0.715   0.197 0.309          3.1


**Observations:**

- F1 peaks at threshold 0.65 (F1 = 0.529, precision = 0.495, recall = 0.568).
- At 0.65, 12.9% of customers get flagged as high confidence, and about half of those actually subscribe.
- The default threshold of 0.50 over-flags (17.4% of customers) and produces too many false positives for a call-center workflow.

But threshold isn't the whole story. Instead of a single yes/no flag, the Caller Companion will surface three tiers based on calibrated probability:

- **High confidence (p >= 0.65):** "Strong likelihood of subscribing. Lean into the conversation."
- **Medium confidence (0.30 <= p < 0.65):** "Some positive signal. Read the brief, use judgment."
- **Low confidence (p < 0.30):** "Model is uncertain. Trust your instincts."

This is a Responsible AI choice as much as a product one. Low-confidence customers get an explicit "the model isn't sure" framing rather than a defaulted-to-yes prediction.

In [15]:
# save thresholds for the app to use
THRESHOLD_HIGH = 0.65
THRESHOLD_MEDIUM = 0.30

print(f"High confidence threshold: {THRESHOLD_HIGH}")
print(f"Medium confidence threshold: {THRESHOLD_MEDIUM}")
print(f"Customers in high confidence: {(test_proba >= THRESHOLD_HIGH).mean() * 100:.1f}%")
print(f"Customers in medium confidence: {((test_proba >= THRESHOLD_MEDIUM) & (test_proba < THRESHOLD_HIGH)).mean() * 100:.1f}%")
print(f"Customers in low confidence: {(test_proba < THRESHOLD_MEDIUM).mean() * 100:.1f}%")

High confidence threshold: 0.65
Medium confidence threshold: 0.3
Customers in high confidence: 12.9%
Customers in medium confidence: 26.7%
Customers in low confidence: 60.4%


## Feature importance

Two things to extract from the trained model:

**Global feature importance** tells me which features the model relies on overall. This goes in the Model Report.

**Per-customer feature contributions** tell me which features pushed each individual prediction up or down. This is what powers the "talking points" the agent sees for each customer in the Caller Companion. A customer flagged as high confidence *because of* prior subscription gets different talking points than one flagged *because of* the macroeconomic context.

In [16]:
# global feature importance from the trained LightGBM model
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 15 features by importance:")
print(feature_importance.head(15).to_string(index=False))

Top 15 features by importance:
       feature  importance
           age        3474
     euribor3m        2945
      campaign        1626
           job         818
       marital         770
       housing         700
cons.price.idx         579
     education         560
 cons.conf.idx         486
   day_of_week         486
          loan         410
         month         391
         pdays         321
       contact         266
      previous         255


**Observations:**

- `age` is the top feature, vindicating the choice of a tree-based model. Linear correlation reported 0.030 for age but LightGBM extracted real signal from the U-shape pattern.
- `euribor3m` is second, reflecting the strong macroeconomic context in the data. The 2008 financial crisis features are doing meaningful predictive work but tie the model to that time period.
- `campaign` (contact count this campaign) is third. Diminishing returns on repeat contacts is a real pattern.
- `poutcome` (prior campaign outcome) is not in the top 15 despite being the strongest *categorical* signal in EDA. This is a quirk of how LightGBM measures importance (by split count, which underweights features that apply to small subsets of customers). The model still uses poutcome effectively for the 3.3% of customers it applies to, the importance score just doesn't reflect that.

For the Caller Companion app, the agent talking points will translate these features into agent-readable language. `age` becomes "demographic profile" framing, `campaign` becomes "contact history" framing, `euribor3m` won't be surfaced directly because it's not actionable for the agent.

## Exporting predictions.csv

This is one of the required submission artifacts. The hackathon brief specifies SML teams must submit a `predictions.csv` for the test set. Including the columns judges need to verify our metrics: the customer index, predicted probability, the binary prediction at our chosen threshold, the confidence band, and the actual label so judges can compute their own metrics.

In [17]:
# build the predictions dataframe
predictions_df = pd.DataFrame({
    'customer_index': X_test.index,
    'predicted_probability': test_proba.round(4),
    'predicted_label': (test_proba >= THRESHOLD_HIGH).astype(int),
    'confidence_band': pd.cut(
        test_proba,
        bins=[-0.01, THRESHOLD_MEDIUM, THRESHOLD_HIGH, 1.01],
        labels=['low', 'medium', 'high']
    ),
    'actual_label': y_test.values
})

# save to model/ folder
predictions_df.to_csv('predictions.csv', index=False)

print(f"Saved predictions.csv with {len(predictions_df):,} rows")
print(f"\nFirst 10 rows:")
print(predictions_df.head(10).to_string(index=False))
print(f"\nDistribution by confidence band:")
print(predictions_df['confidence_band'].value_counts().sort_index())

Saved predictions.csv with 8,238 rows

First 10 rows:
 customer_index  predicted_probability  predicted_label confidence_band  actual_label
          14455                 0.3248                0          medium             0
          36380                 0.8283                1            high             0
          40076                 0.9496                1            high             0
          10778                 0.1388                0             low             0
          27939                 0.8815                1            high             0
          37546                 0.6600                1            high             0
          27310                 0.3996                0          medium             0
           8377                 0.1800                0             low             0
          40138                 0.8984                1            high             1
          20148                 0.3749                0          medium             0


## Saving the model

Exporting three artifacts for the Caller Companion app:

1. `model.pkl` - the trained LightGBM model, ready to load and call `predict_proba()` on
2. `feature_importance.csv` - global importance scores for the Model Report
3. `model_metadata.json` - the thresholds, feature names, and metric values so the app knows how to interpret the model's output without re-deriving them

In [18]:
import json

# save the trained model
joblib.dump(final_model, 'model.pkl')
print("Saved model.pkl")

# save feature importance
feature_importance.to_csv('feature_importance.csv', index=False)
print("Saved feature_importance.csv")

# save metadata the app will need
metadata = {
    'model_type': 'LightGBM',
    'n_estimators': 500,
    'random_state': RANDOM_STATE,
    'thresholds': {
        'high_confidence': THRESHOLD_HIGH,
        'medium_confidence': THRESHOLD_MEDIUM
    },
    'test_metrics': {
        'roc_auc': float(test_roc_auc),
        'pr_auc': float(test_pr_auc),
        'brier_score': float(test_brier)
    },
    'baseline_metrics': {
        'roc_auc': float(baseline_roc_auc),
        'pr_auc': float(baseline_pr_auc),
        'brier_score': float(baseline_brier)
    },
    'feature_names': X_train.columns.tolist(),
    'categorical_features': categorical_cols,
    'numeric_features': numeric_cols,
    'class_distribution': {
        'positive_rate': float(y.mean()),
        'train_size': len(X_train),
        'test_size': len(X_test)
    },
    'dropped_features': ['duration'],
    'drop_reasons': {
        'duration': 'post-call leakage, only known after call ends'
    }
}

with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Saved model_metadata.json")

print(f"\nAll artifacts saved. Files in current directory:")
import os
for f in sorted(os.listdir('.')):
    if not f.startswith('.'):
        size_kb = os.path.getsize(f) / 1024
        print(f"  {f} ({size_kb:.1f} KB)")

Saved model.pkl
Saved feature_importance.csv
Saved model_metadata.json

All artifacts saved. Files in current directory:
  feature_importance.csv (0.3 KB)
  model.pkl (1633.5 KB)
  model_metadata.json (1.3 KB)
  predictions.csv (173.4 KB)
  sample_data (4.0 KB)


## Downloading artifacts to push to the repo

Colab and GitHub aren't directly connected for arbitrary file uploads, so I'm downloading the model artifacts as a zip to add them to the repo manually.

In [19]:
import shutil
from google.colab import files

# bundle the artifacts into a zip
artifacts = ['model.pkl', 'predictions.csv', 'feature_importance.csv', 'model_metadata.json']

with zipfile.ZipFile('model_artifacts.zip', 'w') as zf:
    for artifact in artifacts:
        zf.write(artifact)

# trigger download
files.download('model_artifacts.zip')
print(f"Downloaded model_artifacts.zip with {len(artifacts)} files.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded model_artifacts.zip with 4 files.


## Phase 2 findings

Wrapping up the modeling work with the key takeaways for the Model Report.

1. **Honest ROC-AUC of 0.8008.** Tied with the baseline logistic regression. Without the leaky `duration` feature, the dataset has a natural ceiling around 0.80 that no amount of hyperparameter tuning could break.

2. **LightGBM wins on PR-AUC and Brier.** ROC-AUC tied, but LightGBM improved PR-AUC from 0.46 to 0.48 and Brier from 0.16 to 0.13. The product needs trustworthy probabilities, not just rankings, so LightGBM is the production choice.

3. **Three-tier confidence framing.** Instead of a binary yes/no flag at 0.50, the Caller Companion uses three tiers: high confidence (p >= 0.65, 12.9% of customers), medium confidence (0.30 to 0.65, 26.7%), and low confidence (p < 0.30, 60.4%). Low confidence explicitly tells the agent "trust your instincts, the model isn't sure."

4. **Top features.** `age` (validated the U-shape from EDA, a tree-based win), `euribor3m` (macroeconomic crisis signal), `campaign` (contact frequency), `job`, and `marital` are the top 5 by importance. Worth flagging that `poutcome` doesn't appear in the top 15 by split count, but it's still the strongest categorical signal for the 3.3% of customers it applies to.

5. **Generalization concerns.** The macroeconomic features tie this model to 2008-2010 Portugal. The model trained today on bank marketing data would not generalize to solar leads in California or B2B SaaS outbound. The *product pattern* (calibrated brief, talking points, confidence band, outcome logging) does generalize, but the model itself is bound to its training context.